# Receive and validate the Sentinel Azure resume bundle

Run this notebook **on the Azure VM** from VS Code, with the Python 3.12 environment/kernel where `pandas`, `pyarrow`, and `numpy` are installed. It verifies the transferred `.tar.gz` against its `.sha256` file, checks the archive layout and free-space impact, extracts the existing inventory/mapping/NPZ checkpoints, and validates the result.

It does **not** start processing, rebuild the inventory, delete the archive, or delete any existing Sentinel files. The extraction cell deliberately stops if files in the bundle would overwrite existing files; inspect that situation before proceeding.

In [1]:
from pathlib import Path, PurePosixPath
import hashlib
import json
import os
import shutil
import sqlite3
import tarfile
import re

import numpy as np
import pandas as pd

PROJECT_ROOT = (Path.home() / "sentinel1-rtc").resolve()
EXPECTED_REL = Path("Data/Global/pipeline_stage1/sentinel1")
EXPECTED_PREFIX = EXPECTED_REL.as_posix() + "/"
INVENTORY_REL = EXPECTED_REL / "scene_inventory_nearest_v1.sqlite"
MAPPING_REL = EXPECTED_REL / "stations_with_clusters.parquet"
SHARDS_REL = EXPECTED_REL / "scene_observations_nearest_overview_v1"

# Keep at least 10 GiB free after extraction; this leaves room above the
# pipeline's 8 GiB hard stop threshold.
MIN_FREE_AFTER_EXTRACT_GIB = 10.0

print("Project root:", PROJECT_ROOT)
print("Exists:", PROJECT_ROOT.is_dir())
print("Current free space (GiB):", round(shutil.disk_usage(PROJECT_ROOT).free / 1024**3, 2))
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(f"Project root not found: {PROJECT_ROOT}")


Project root: /home/Mayur/sentinel1-rtc
Exists: True
Current free space (GiB): 23.9


## 1. Find the transfer files and verify the checksum

Place the `.tar.gz` archive and its matching `.tar.gz.sha256` file directly in `~/sentinel1-rtc`. Run this cell before extraction.

In [2]:
checksum_files = sorted(PROJECT_ROOT.glob("sentinel1_azure_resume_bundle_*.tar.gz.sha256"))
if len(checksum_files) != 1:
    raise RuntimeError(
        f"Expected exactly one sentinel1_azure_resume_bundle_*.tar.gz.sha256 file in {PROJECT_ROOT}; "
        f"found {len(checksum_files)}: {[p.name for p in checksum_files]}"
    )
CHECKSUM_PATH = checksum_files[0]
line = CHECKSUM_PATH.read_text(encoding="utf-8").strip()
parts = line.split(maxsplit=1)
if len(parts) != 2:
    raise ValueError(f"Unexpected checksum-file format: {line!r}")
expected_sha, named_archive = parts
named_archive = named_archive.lstrip("* ")
if Path(named_archive).name != named_archive:
    raise ValueError(f"Checksum references an unexpected path: {named_archive!r}")
ARCHIVE_PATH = PROJECT_ROOT / named_archive
if not ARCHIVE_PATH.is_file():
    raise FileNotFoundError(f"Archive named by checksum is missing: {ARCHIVE_PATH}")

h = hashlib.sha256()
with ARCHIVE_PATH.open("rb") as f:
    for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
        h.update(block)
actual_sha = h.hexdigest()
print("Archive:", ARCHIVE_PATH.name)
print("Archive size (GiB):", round(ARCHIVE_PATH.stat().st_size / 1024**3, 3))
print("Checksum file:", CHECKSUM_PATH.name)
print("Expected SHA-256:", expected_sha)
print("Actual SHA-256:  ", actual_sha)
if actual_sha.lower() != expected_sha.lower():
    raise RuntimeError("CHECKSUM MISMATCH: do not extract; recopy the archive and checksum.")
print("CHECKSUM: PASS")


Archive: sentinel1_azure_resume_bundle_20261010_133846.tar.gz
Archive size (GiB): 0.22
Checksum file: sentinel1_azure_resume_bundle_20261010_133846.tar.gz.sha256
Expected SHA-256: 7d76ee8147458a53be31e421a58ba0ee63974d94ba19dc539cdf2a6efa22b803
Actual SHA-256:   7d76ee8147458a53be31e421a58ba0ee63974d94ba19dc539cdf2a6efa22b803
CHECKSUM: PASS


## 2. Audit archive contents, size, and destination conflicts

This cell reads archive metadata only. It does not extract anything.

In [3]:
with tarfile.open(ARCHIVE_PATH, mode="r:gz") as tf:
    MEMBERS = tf.getmembers()

if not MEMBERS:
    raise RuntimeError("Archive is empty.")

allowed_names = set()
regular_members = []
unsafe = []
for member in MEMBERS:
    p = PurePosixPath(member.name)
    if p.is_absolute() or ".." in p.parts or not member.name.startswith(EXPECTED_PREFIX):
        unsafe.append(member.name)
        continue
    if not member.isfile():
        # The bundle creator adds files only, so unexpected links/dirs are rejected.
        unsafe.append(member.name)
        continue
    regular_members.append(member)
    allowed_names.add(member.name)
if unsafe:
    raise RuntimeError(f"Unexpected/unsafe archive members (first 10): {unsafe[:10]}")

required = {
    INVENTORY_REL.as_posix(),
    MAPPING_REL.as_posix(),
    (EXPECTED_REL / "azure_resume_manifest.json").as_posix(),
}
missing_required = required - allowed_names
if missing_required:
    raise RuntimeError(f"Required files missing from archive: {sorted(missing_required)}")

shard_prefix = SHARDS_REL.as_posix() + "/"
archive_shard_members = [m for m in regular_members if m.name.startswith(shard_prefix) and m.name.endswith(".npz")]
manifest_member = next(m for m in regular_members if m.name == (EXPECTED_REL / "azure_resume_manifest.json").as_posix())
with tarfile.open(ARCHIVE_PATH, mode="r:gz") as tf:
    manifest = json.load(tf.extractfile(manifest_member))

payload_bytes = sum(m.size for m in regular_members)
free_now = shutil.disk_usage(PROJECT_ROOT).free
projected_free = free_now - payload_bytes
print("Archive files:", f"{len(regular_members):,}")
print("Scene shards in archive:", f"{len(archive_shard_members):,}")
print("Uncompressed payload (GiB):", round(payload_bytes / 1024**3, 3))
print("Free now (GiB):", round(free_now / 1024**3, 2))
print("Projected free after extraction (GiB):", round(projected_free / 1024**3, 2))
print("Manifest summary:", manifest)
if projected_free < MIN_FREE_AFTER_EXTRACT_GIB * 1024**3:
    raise RuntimeError(
        f"Not extracting: projected free space is {projected_free / 1024**3:.2f} GiB; "
        f"required minimum is {MIN_FREE_AFTER_EXTRACT_GIB:.1f} GiB."
    )

# Fail closed rather than overwriting existing database/mapping/shard files.
existing_conflicts = []
for member in regular_members:
    rel = Path(*PurePosixPath(member.name).parts)
    dest = (PROJECT_ROOT / rel).resolve()
    if not dest.is_relative_to(PROJECT_ROOT):
        raise RuntimeError(f"Member escapes project root: {member.name}")
    if dest.exists():
        existing_conflicts.append(str(dest.relative_to(PROJECT_ROOT)))
if existing_conflicts:
    raise RuntimeError(
        "Extraction stopped to avoid overwriting existing files. "
        f"First conflicts: {existing_conflicts[:20]} (total {len(existing_conflicts)}). "
        "Do not delete them blindly; report this output before continuing."
    )

print("ARCHIVE AUDIT: PASS — safe paths, required files present, no file conflicts.")
print("If the summary is correct, set EXTRACT_BUNDLE_NOW=True in the next cell and run it.")


Archive files: 21,658
Scene shards in archive: 21,655
Uncompressed payload (GiB): 1.468
Free now (GiB): 23.9
Projected free after extraction (GiB): 22.43
Manifest summary: {'created_utc': '2026-10-10T13:38:50.277946+00:00', 'project_relative_sentinel_path': 'Data/Global/pipeline_stage1/sentinel1', 'inventory_filename': 'scene_inventory_nearest_v1.sqlite', 'cluster_mapping_filename': 'stations_with_clusters.parquet', 'scene_shard_directory': 'scene_observations_nearest_overview_v1', 'inventory_scene_count': 204033, 'cluster_searches_completed': 819, 'cluster_search_count': 819, 'scene_cluster_link_rows': 528280, 'clusters_with_scene_links': 818, 'scenes_without_cluster_links': 0, 'existing_npz_shards': 21655, 'included_stac_caches': False, 'included_legacy_yearly_zarr': False}
ARCHIVE AUDIT: PASS — safe paths, required files present, no file conflicts.
If the summary is correct, set EXTRACT_BUNDLE_NOW=True in the next cell and run it.


## 3. Extract the verified bundle (manual switch)

This cell is intentionally disabled. Read the audit above first, then change `EXTRACT_BUNDLE_NOW` to `True` and run this cell. It only extracts the three resume components, the NPZ shards, and the manifest.

In [6]:
EXTRACT_BUNDLE_NOW = True  # Change to True only after the audit above passes.

if not EXTRACT_BUNDLE_NOW:
    print("Extraction disabled. Set EXTRACT_BUNDLE_NOW=True after reviewing the audit.")
else:
    # Recheck checksum and conflicts immediately before extraction.
    h = hashlib.sha256()
    with ARCHIVE_PATH.open("rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    current_line = CHECKSUM_PATH.read_text(encoding="utf-8").strip().split(maxsplit=1)
    if h.hexdigest().lower() != current_line[0].lower():
        raise RuntimeError("Archive checksum changed after audit; refusing extraction.")

    with tarfile.open(ARCHIVE_PATH, mode="r:gz") as tf:
        members = tf.getmembers()
        for member in members:
            p = PurePosixPath(member.name)
            if p.is_absolute() or ".." in p.parts or not member.isfile() or not member.name.startswith(EXPECTED_PREFIX):
                raise RuntimeError(f"Unexpected/unsafe member: {member.name!r}")
            dest = (PROJECT_ROOT / Path(*p.parts)).resolve()
            if not dest.is_relative_to(PROJECT_ROOT) or dest.exists():
                raise RuntimeError(f"Destination conflict/unsafe path: {dest}")
        for idx, member in enumerate(members, start=1):
            dest = (PROJECT_ROOT / Path(*PurePosixPath(member.name).parts)).resolve()
            dest.parent.mkdir(parents=True, exist_ok=True)
            src = tf.extractfile(member)
            if src is None:
                raise RuntimeError(f"Could not read archive member: {member.name}")
            # Exclusive create avoids clobbering files if anything changed after preflight.
            with src, dest.open("xb") as out:
                shutil.copyfileobj(src, out, length=8 * 1024 * 1024)
            if idx % 1000 == 0 or idx == len(members):
                print(f"Extracted {idx:,}/{len(members):,} files")
    print("Extraction finished. Run the validation cell next.")


Extracted 1,000/21,658 files
Extracted 2,000/21,658 files
Extracted 3,000/21,658 files
Extracted 4,000/21,658 files
Extracted 5,000/21,658 files
Extracted 6,000/21,658 files
Extracted 7,000/21,658 files
Extracted 8,000/21,658 files
Extracted 9,000/21,658 files
Extracted 10,000/21,658 files
Extracted 11,000/21,658 files
Extracted 12,000/21,658 files
Extracted 13,000/21,658 files
Extracted 14,000/21,658 files
Extracted 15,000/21,658 files
Extracted 16,000/21,658 files
Extracted 17,000/21,658 files
Extracted 18,000/21,658 files
Extracted 19,000/21,658 files
Extracted 20,000/21,658 files
Extracted 21,000/21,658 files
Extracted 21,658/21,658 files
Extraction finished. Run the validation cell next.


## 4. Validate the extracted inventory, cluster mapping, and scene shards

This cell is read-only. It confirms the expected inventory counts and verifies that filenames for the existing NPZ shards correspond to IDs in the database.

In [7]:
inventory_path = PROJECT_ROOT / INVENTORY_REL
mapping_path = PROJECT_ROOT / MAPPING_REL
shards_dir = PROJECT_ROOT / SHARDS_REL
manifest_path = PROJECT_ROOT / EXPECTED_REL / "azure_resume_manifest.json"
for path in (inventory_path, mapping_path, shards_dir, manifest_path):
    if not path.exists():
        raise FileNotFoundError(f"Expected extracted item is missing: {path}")

with sqlite3.connect(f"file:{inventory_path.resolve().as_posix()}?mode=ro", uri=True, timeout=30) as con:
    n_scenes = con.execute("SELECT COUNT(*) FROM scenes").fetchone()[0]
    completed = con.execute("SELECT COUNT(*) FROM cluster_searches WHERE completed=1").fetchone()[0]
    n_clusters = con.execute("SELECT COUNT(*) FROM cluster_searches").fetchone()[0]
    links = con.execute("SELECT COUNT(*) FROM scene_clusters").fetchone()[0]
    scenes_unlinked = con.execute("""SELECT COUNT(*) FROM scenes s LEFT JOIN scene_clusters sc ON sc.scene_id=s.scene_id WHERE sc.scene_id IS NULL""").fetchone()[0]

mapping = pd.read_parquet(mapping_path)
if not {"station_id", "cluster_id"}.issubset(mapping.columns):
    raise RuntimeError(f"Unexpected mapping columns: {list(mapping.columns)}")
station_count = int(mapping["station_id"].nunique())
cluster_count = int(mapping["cluster_id"].nunique())
shard_files = sorted(p for p in shards_dir.glob("*.npz") if p.is_file())
actual_names = {p.name for p in shard_files}

def shard_name(scene_id):
    sid = str(scene_id)
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", sid)
    suffix = hashlib.sha1(sid.encode("utf-8")).hexdigest()[:10]
    return f"{safe[:140]}_{suffix}.npz"

matches = 0
missing_shards = 0
with sqlite3.connect(f"file:{inventory_path.resolve().as_posix()}?mode=ro", uri=True, timeout=30) as con:
    for (scene_id,) in con.execute("SELECT scene_id FROM scenes"):
        if shard_name(scene_id) in actual_names:
            matches += 1
        else:
            missing_shards += 1

print("Unique inventory scenes:", f"{n_scenes:,}")
print("Completed cluster searches:", f"{completed:,}/{n_clusters:,}")
print("Scene-cluster links:", f"{links:,}")
print("Scenes without cluster links:", f"{scenes_unlinked:,}")
print("Stations in saved mapping:", f"{station_count:,}")
print("Clusters in saved mapping:", f"{cluster_count:,}")
print("Extracted NPZ shards:", f"{len(shard_files):,}")
print("Existing shards matching inventory IDs:", f"{matches:,}")
print("Inventory scenes without shards (expected to be pending):", f"{missing_shards:,}")
print("Manifest:", json.loads(manifest_path.read_text(encoding="utf-8")))

if n_scenes != 204033 or completed != 819 or n_clusters != 819:
    raise RuntimeError("Inventory counts differ from the audited local inventory; do not start processing.")
if scenes_unlinked != 0:
    raise RuntimeError("Some scenes have no cluster links; do not start processing.")
if station_count != 2210 or cluster_count != 819:
    raise RuntimeError("Saved cluster mapping count differs from the audited local mapping.")
if matches != len(shard_files):
    raise RuntimeError("At least one extracted shard filename did not match an inventory scene ID.")

# Open up to three shards to verify NPZ integrity without loading every shard fully.
for p in shard_files[:3]:
    with np.load(p, allow_pickle=False) as z:
        print("Readable shard:", p.name, "keys=", list(z.files), "data shape=", z["data"].shape)
print("VALIDATION PASS — ready to open the resume notebook; this notebook has not started the pipeline.")


Unique inventory scenes: 204,033
Completed cluster searches: 819/819
Scene-cluster links: 528,280
Scenes without cluster links: 0
Stations in saved mapping: 2,210
Clusters in saved mapping: 819
Extracted NPZ shards: 21,655
Existing shards matching inventory IDs: 21,655
Inventory scenes without shards (expected to be pending): 182,378
Manifest: {'created_utc': '2026-10-10T13:38:50.277946+00:00', 'project_relative_sentinel_path': 'Data/Global/pipeline_stage1/sentinel1', 'inventory_filename': 'scene_inventory_nearest_v1.sqlite', 'cluster_mapping_filename': 'stations_with_clusters.parquet', 'scene_shard_directory': 'scene_observations_nearest_overview_v1', 'inventory_scene_count': 204033, 'cluster_searches_completed': 819, 'cluster_search_count': 819, 'scene_cluster_link_rows': 528280, 'clusters_with_scene_links': 818, 'scenes_without_cluster_links': 0, 'existing_npz_shards': 21655, 'included_stac_caches': False, 'included_legacy_yearly_zarr': False}
Readable shard: S1A_IW_GRDH_1SDH_201503

## Next: resume the existing pipeline

1. Do **not** delete the WSL originals.
2. Open `BUILD_SENTINEL_RTC_GLOBAL_v9_AZURE_RESUME_EXISTING.ipynb` on this Azure VM.
3. Restart its kernel, then run the setup/definition cells from top to bottom while all global run flags remain `False` (you may use Run All only in this disabled state).
4. Confirm it prints the existing inventory at `Data/Global/pipeline_stage1/sentinel1/scene_inventory_nearest_v1.sqlite`, the shard path `scene_observations_nearest_overview_v1`, and 4 scene workers.
5. In the final run-control cell only, set `RUN_RESUME_EXISTING_PIPELINE = True` (leave the other flags `False`) and run that cell. Do not use Run All after enabling it.

The resume pipeline skips already-existing scene shard files and then continues nearest-valid yearly finalization. Its disk thresholds are 10 GiB warning and 8 GiB stop. On a stop, retain the SQLite inventory and shards, manually pull/download the files you need, free enough VM disk space, and rerun the final run-control cell to resume.